# OCR-based Image Classification

This notebook extracts text from images using OCR (pytesseract) and trains a text classifier (TF-IDF + Logistic Regression) to predict the image class. It includes data collection, OCR extraction, model training, evaluation, inference, and artifact saving.

In [1]:
# Install required Python packages if missing (won't install system Tesseract)
import importlib, subprocess, sys
def ensure_package(pkg, import_name=None):
    m = import_name or pkg
    try:
        importlib.import_module(m)
        print(f'OK: {pkg} available')
    except ModuleNotFoundError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

ensure_package('pytesseract')
ensure_package('pillow', 'PIL')
ensure_package('scikit-learn', 'sklearn')
ensure_package('joblib')
ensure_package('pandas')
ensure_package('tqdm')
ensure_package('numpy')

Installing pytesseract...
OK: pillow available
OK: scikit-learn available
OK: joblib available
OK: pandas available
Installing tqdm...
OK: numpy available


In [2]:
# Imports
from pathlib import Path
from PIL import Image
import pytesseract
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from joblib import dump, load
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuration - workspace-relative paths
ROOT = Path('..').resolve()  # notebook is in TFM/ folder, Data is at workspace root
DATA_DIR = (ROOT / 'Data')
MODELS_DIR = Path('Models')
MODELS_DIR.mkdir(exist_ok=True)
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')
print('Data dir:', DATA_DIR)
print('Models dir:', MODELS_DIR.resolve())

Data dir: C:\Users\Lenovo\Desktop\UB\TFM\Data
Models dir: C:\Users\Lenovo\Desktop\UB\TFM\TFM\Models


In [4]:
# Collect top-level class image paths (same convention as other notebooks)
def collect_top_level_samples(root_dir):
    root = Path(root_dir)
    classes = sorted([p.name for p in root.iterdir() if p.is_dir()])
    samples = []
    for idx, cname in enumerate(classes):
        for img in (root / cname).rglob('*'):
            if img.is_file() and img.suffix.lower() in IMAGE_EXTENSIONS:
                samples.append((str(img), cname))
    return classes, samples

classes, samples = collect_top_level_samples(DATA_DIR)
print('Found classes:', classes)
print('Total images found:', len(samples))

Found classes: ['Banner aplicación', 'Cierre aplicación', 'Error aplicativo', 'Error funcional', 'Error terminal', 'Indeterminado', 'Revisión circuito', 'Timeout']
Total images found: 2764


In [8]:
# Quick Tesseract availability check
try:
    print('pytesseract version:', pytesseract.get_tesseract_version())
except Exception as e:
    # Try to locate a common install path and configure pytesseract to use it
    import os
    tesseract_path = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    if os.path.exists(tesseract_path):
        pytesseract.pytesseract.tesseract_cmd = tesseract_path
        try:
            print('pytesseract version:', pytesseract.get_tesseract_version())
        except Exception as e2:
            print('Tesseract found but pytesseract still cannot access it:', e2)
    else:
        print('Warning: Tesseract not available or not found by pytesseract. You must install Tesseract-OCR separately for OCR to work.')
        print('On Windows, install from https://github.com/tesseract-ocr/tesseract and set pytesseract.pytesseract.tesseract_cmd if needed.')


pytesseract version: 5.5.0.20241111


In [9]:
# Build a dataframe with OCR-extracted text and labels (this can be slow)
def extract_text_from_image(path):
    try:
        img = Image.open(path).convert('RGB')
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return ''

records = []
for img_path, label in tqdm(samples, desc='OCR pass'):
    txt = extract_text_from_image(img_path)
    records.append({'path': img_path, 'label': label, 'text': txt})

df = pd.DataFrame(records)
# Optionally drop empty-text rows or keep them (they may still carry signal)
print('Samples with empty OCR text:', (df['text']=="").sum())
df.head()

OCR pass: 100%|██████████| 2764/2764 [39:30<00:00,  1.17it/s]

Samples with empty OCR text: 6


,path,label,text
0,C:\Users\Lenovo\Desktop\UB\TFM\Data\Banner apl...,Banner aplicación,© 8 Fu\n\nInformacion sobre los\nDatos mostrad...
1,C:\Users\Lenovo\Desktop\UB\TFM\Data\Banner apl...,Banner aplicación,Informacion sobre los Datos mostrados\nen Caix...
2,C:\Users\Lenovo\Desktop\UB\TFM\Data\Banner apl...,Banner aplicación,100%@\n\nInformacion sobre los\nDatos mostrado...
3,C:\Users\Lenovo\Desktop\UB\TFM\Data\Banner apl...,Banner aplicación,23:21 Mt\n\nInformacion sobre los\nDatos mostr...
4,C:\Users\Lenovo\Desktop\UB\TFM\Data\Banner apl...,Banner aplicación,5:42 BOR = @ ull 100% &\n\nActualiza la app\n\...


In [10]:
# Prepare dataset for text classification
X = df['text'].fillna('')
y = df['label']

# Train/val split stratified by label
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train samples:', len(X_train), 'Val samples:', len(X_val))

# Vectorize text
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_t = vectorizer.fit_transform(X_train)
X_val_t = vectorizer.transform(X_val)

Train samples: 2211 Val samples: 553


In [12]:
# Train a simple Logistic Regression classifier
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_t, y_train)

# Evaluate
y_pred = clf.predict(X_val_t)
print('Accuracy:', accuracy_score(y_val, y_pred))
print('\nClassification report:')
print(classification_report(y_val, y_pred))


Accuracy: 0.9602169981916817

Classification report:
                   precision    recall  f1-score   support

Banner aplicación       1.00      0.95      0.97        20
Cierre aplicación       0.96      0.81      0.88        31
 Error aplicativo       1.00      0.97      0.99        36
  Error funcional       0.95      0.97      0.96       107
   Error terminal       1.00      1.00      1.00        81
    Indeterminado       0.00      0.00      0.00         3
Revisión circuito       0.97      0.98      0.98       236
          Timeout       0.83      0.90      0.86        39

         accuracy                           0.96       553
        macro avg       0.84      0.82      0.83       553
     weighted avg       0.96      0.96      0.96       553



In [13]:
# Save artifacts: vectorizer + classifier
dump(vectorizer, MODELS_DIR / 'ocr_vectorizer.joblib')
dump(clf, MODELS_DIR / 'ocr_text_classifier.joblib')
df.to_csv(MODELS_DIR / 'ocr_samples.csv', index=False)
print('Saved vectorizer and classifier to', MODELS_DIR)

Saved vectorizer and classifier to Models


In [14]:
# Inference helper: given an image, extract text and predict class
def predict_image_class_from_ocr(image_path, vectorizer_path=MODELS_DIR / 'ocr_vectorizer.joblib', clf_path=MODELS_DIR / 'ocr_text_classifier.joblib', confidence_threshold=0.5):
    vec = load(vectorizer_path)
    model = load(clf_path)
    txt = extract_text_from_image(image_path)
    if not txt:
        return {'predicted': 'unknown', 'confidence': 0.0, 'text': ''}
    vt = vec.transform([txt])
    probs = model.predict_proba(vt)[0]
    idx = probs.argmax()
    label = model.classes_[idx]
    confidence = float(probs[idx])
    if confidence < confidence_threshold:
        return {'predicted': 'uncertain', 'confidence': confidence, 'text': txt}
    return {'predicted': label, 'confidence': confidence, 'text': txt}

# Example: predict on first validation sample if available
if len(X_val) > 0:
    example_path = df.loc[X_val.index[0], 'path']
    print('Example path:', example_path)
    print(predict_image_class_from_ocr(example_path))

Example path: C:\Users\Lenovo\Desktop\UB\TFM\Data\Revisión circuito\Transacciones\20260504223854_100165.png
{'predicted': 'Revisión circuito', 'confidence': 0.8564032082635514, 'text': '22:42 m@O@ ° KX Aull 3%\n\n< ®\nCuenta *0047\n\nViernes 19\n€< QuotaT. Visa E- -0,24€\n>\np »\n0,00€\nMiércoles 17\n<  Ven.ac. +0,23€\n=> :\nCaixabank »\n0,24€\n< _ Traspaso Propio +0,01€\n“2 0,01€ ?\n<  Traspaso -0,01€\n— 0,00€ >\n<  Traspaso +0,01€\n2 0,01€ ?\nNoviembre 2025\nMiércoles 26\n< Cuota DiaA Dia -0,10€\n- 0,00€ ?\n\nIII O <'}


---
Notes:
- This approach relies on Tesseract-OCR being installed on your system. On Windows, install Tesseract and set `pytesseract.pytesseract.tesseract_cmd` to the tesseract executable path if needed.
- If OCR text is sparse or unreliable, consider a hybrid approach: combine image-based CNN features with OCR text features, or train an end-to-end image classifier.

In [15]:
# Write a minimal requirements.txt for this notebook
reqs = ['pytesseract', 'pillow', 'scikit-learn', 'joblib', 'pandas', 'tqdm', 'numpy']
with open('requirements_ocr.txt', 'w') as f:
    f.write('\n'.join(reqs))
print('requirements_ocr.txt created')


requirements_ocr.txt created
